Phase 3.1 — Build our first model

For our first experiment, we'll use MobileNetV2 with transfer learning.

Why?

It's relatively lightweight, fast to train, and suitable for getting our first complete experiment working. After that, we can compare it against EfficientNetB0 and ResNet50.

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [11]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomTranslation(
        height_factor=0.05,
        width_factor=0.05
    ),
    tf.keras.layers.RandomContrast(0.10)
], name="data_augmentation")

print("Data augmentation ready.")

Data augmentation ready.


In [12]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

DATA_DIR = "../data/raw/chest_xray"
TRAIN_DIR = f"{DATA_DIR}/train"

In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="binary",
    color_mode="rgb",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.20,
    subset="training",
    seed=SEED,
    shuffle=True
)

Found 5216 files belonging to 2 classes.
Using 4173 files for training.


In [4]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="binary",
    color_mode="rgb",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    shuffle=False
)

Found 5216 files belonging to 2 classes.
Using 1043 files for validation.


In [5]:
print("Classes:", train_ds.class_names)

for images, labels in train_ds.take(1):
    print("Images:", images.shape)
    print("Labels:", labels.shape)

Classes: ['NORMAL', 'PNEUMONIA']
Images: (32, 224, 224, 3)
Labels: (32, 1)


Build MobileNetV2

In [6]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


MobileNetV2
     ↓
ImageNet pretrained weights
     ↓
Freeze the pretrained layers
     ↓
Add our own classification head

In [13]:
inputs = keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Step 1 — Add class weights

In [14]:
class_weight = {
    0: 1.945,
    1: 0.673
}

print(class_weight)

{0: 1.945, 1: 0.673}


MODEL COMPILING

In [15]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall")
    ]
)

CREATING CALLBACK

In [16]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        "../models/best_mobilenetv2.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("Callbacks ready.")


Callbacks ready.


Optimize datasets

In [17]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

START TRAINING

In [18]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weight,
    callbacks=callbacks
)

Epoch 1/20


c:\Users\Priyansh\Documents\ChestVision-AI\venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 305ms/step - accuracy: 0.5495 - loss: 0.7370 - precision: 0.7823 - recall: 0.5401
Epoch 1: val_loss improved from None to 0.37149, saving model to ../models/best_mobilenetv2.keras

Epoch 1: finished saving model to ../models/best_mobilenetv2.keras
131/131 ━━━━━━━━━━━━━━━━━━━━ 54s 384ms/step - accuracy: 0.5495 - loss: 0.7370 - precision: 0.7823 - recall: 0.5401 - val_accuracy: 0.9521 - val_loss: 0.3715 - val_precision: 1.0000 - val_recall: 0.9521 - learning_rate: 1.0000e-04
Epoch 2/20
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 305ms/step - accuracy: 0.7067 - loss: 0.5683 - precision: 0.8704 - recall: 0.7082
Epoch 2: val_loss improved from 0.37149 to 0.27183, saving model to ../models/best_mobilenetv2.keras

Epoch 2: finished saving model to ../models/best_mobilenetv2.keras
131/131 ━━━━━━━━━━━━━━━━━━━━ 50s 376ms/step - accuracy: 0.7067 - loss: 0.5683 - precision: 0.8704 - recall: 0.7082 - val_accuracy: 0.9674 - val_loss: 0.2718 - val_precision: 1.0000 - val_recall: 0

KeyboardInterrupt: 